
# YOLO Practical Exercises

This notebook contains:
- Conceptual answers
- YOLOv3 object detection on an image
- Webcam detection (optional)
- YOLOv8 comparison



## Exercise 1: Conceptual Answers

**1. Anchor Boxes**
Anchor boxes provide predefined shapes, allowing the network to predict offsets instead of full box dimensions. This improves speed and helps specialization (e.g., detecting tall vs wide objects).

**2. NMS Threshold**
- High threshold (0.9): keeps many overlapping boxes → duplicates
- Low threshold (0.1): removes too many boxes → may miss objects

**3. Single Shot Trade-off**
YOLO processes detection in one pass, making it fast but less precise for small objects compared to two-stage detectors (which refine proposals).


In [1]:

!pip install opencv-python numpy ultralytics



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import cv2
import numpy as np

# Load class names
with open("coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

# Load YOLO
net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")

# Load image
image = cv2.imread("image.jpg")
height, width = image.shape[:2]

# Create blob
blob = cv2.dnn.blobFromImage(image, 1/255.0, (416, 416), swapRB=True, crop=False)
net.setInput(blob)

# Forward pass
layer_names = net.getUnconnectedOutLayersNames()
outputs = net.forward(layer_names)

boxes = []
confidences = []
class_ids = []

# Process detections
for output in outputs:
    for detection in output:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]
        
        if confidence > 0.5:
            center_x = int(detection[0] * width)
            center_y = int(detection[1] * height)
            w = int(detection[2] * width)
            h = int(detection[3] * height)

            x = int(center_x - w / 2)
            y = int(center_y - h / 2)

            boxes.append([x, y, w, h])
            confidences.append(float(confidence))
            class_ids.append(class_id)

# Apply NMS
indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

# Draw boxes
for i in range(len(boxes)):
    if i in indexes:
        x, y, w, h = boxes[i]
        label = str(classes[class_ids[i]])
        conf = str(round(confidences[i], 2))
        cv2.rectangle(image, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(image, label + " " + conf, (x, y+30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

cv2.imshow("Image", image)
cv2.waitKey(0)
cv2.destroyAllWindows()


FileNotFoundError: [Errno 2] No such file or directory: 'coco.names'

In [ ]:

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    height, width = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416,416), swapRB=True, crop=False)
    net.setInput(blob)
    outputs = net.forward(layer_names)

    boxes, confidences, class_ids = [], [], []

    for output in outputs:
        for detection in output:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]

            if confidence > 0.5:
                center_x = int(detection[0] * width)
                center_y = int(detection[1] * height)
                w = int(detection[2] * width)
                h = int(detection[3] * height)

                x = int(center_x - w/2)
                y = int(center_y - h/2)

                boxes.append([x,y,w,h])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

    for i in range(len(boxes)):
        if i in indexes:
            x,y,w,h = boxes[i]
            label = str(classes[class_ids[i]])
            cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
            cv2.putText(frame,label,(x,y+30),
                        cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

    cv2.imshow("Webcam", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model("image.jpg", show=True)
